<a href="https://colab.research.google.com/github/nig414/AML-experiments/blob/main/AML_Experiment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Load the dataset
print("Please upload the 'iris_synthetic_data.csv' file:")
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

In [ ]:
# 2. Preprocess the dataset
df = df.fillna(df.median(numeric_only=True))
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

X = df.drop(['label', 'label_encoded'], axis=1)
y = df['label_encoded']

# PCA and LDA are sensitive to scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# 3. Apply PCA (Unsupervised)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])

In [ ]:
# 4. Apply LDA (Supervised)
lda = LDA(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)
df_lda = pd.DataFrame(X_lda, columns=['LD1', 'LD2'])

# Display reduced feature heads
print("\n--- PCA Reduced Features (Top 5) ---")
print(df_pca.head())
print("\n--- LDA Reduced Features (Top 5) ---")
print(df_lda.head())

In [ ]:
# 5. Compare Classification Accuracy
def evaluate_model(X_data, y_data, name):
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)
    clf = LogisticRegression()
    clf.fit(X_train, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test))
    print(f"Accuracy with {name}: {acc:.4f}")
    return acc

print("\n--- Accuracy Comparison ---")
acc_orig = evaluate_model(X_scaled, y, "Original Features (4)")
acc_pca = evaluate_model(X_pca, y, "PCA Features (2)")
acc_lda = evaluate_model(X_lda, y, "LDA Features (2)")

In [ ]:
# 6. Plot the 2D Projections
plt.figure(figsize=(14, 6))

# PCA Plot
plt.subplot(1, 2, 1)
for i, target_name in enumerate(le.classes_):
    plt.scatter(X_pca[y == i, 0], X_pca[y == i, 1], label=target_name, alpha=0.7)
plt.title('PCA: Unsupervised Projection')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()

# LDA Plot
plt.subplot(1, 2, 2)
for i, target_name in enumerate(le.classes_):
    plt.scatter(X_lda[y == i, 0], X_lda[y == i, 1], label=target_name, alpha=0.7)
plt.title('LDA: Supervised Projection')
plt.xlabel('Linear Discriminant 1')
plt.ylabel('Linear Discriminant 2')
plt.legend()

plt.tight_layout()
plt.show()